# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) định vị hư hại mặt đường

Đọc `README.md` và `GUIDE.md` trước. Notebook đã triển khai E0–E4. Chạy từ đầu trên Kaggle Notebook với Internet và GPU T4; thêm Kaggle Secrets `KAGGLE_USERNAME` và `KAGGLE_KEY` trước khi chạy.


## 0. Môi trường (Kaggle Notebook, GPU T4)


In [ ]:
# Kaggle/Colab: lấy source hỗ trợ từ repo công khai nếu notebook được upload riêng. Bật Internet và chọn GPU trước khi chạy.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/vuog23/K4-Track-4-Day10-VisionLanguageModel.git"
REPO_NAME = "K4-Track-4-Day10-VisionLanguageModel"
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts" / "prepare_data.py").is_file():
    WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
    PROJECT_ROOT = WORK_ROOT / REPO_NAME
    if not (PROJECT_ROOT / "scripts" / "prepare_data.py").is_file():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements-colab.txt")], check=True)
print("Project root:", PROJECT_ROOT)


In [ ]:
# Check the libraries used by E0-E4 before inference or training.
from importlib.metadata import PackageNotFoundError, version
import torch
import torchvision
import yaml
from PIL import Image, ImageDraw
from bitsandbytes.nn import Linear4bit
from peft import LoraConfig, get_peft_model
from peft.utils.save_and_load import set_peft_model_state_dict
from safetensors.torch import load_file
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration, get_cosine_schedule_with_warmup

required_versions = {
    "transformers": "5.19.0",
    "peft": "0.21.2",
    "bitsandbytes": "0.50.2",
    "accelerate": "1.15.0",
}
for package, expected in required_versions.items():
    actual = version(package)
    if actual != expected:
        raise RuntimeError(f"{package}={actual}; expected {expected}")
version("kaggle")
if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access the Kaggle GPU")
print("Dependency imports OK; CUDA:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
try:
    torchao_version = version("torchao")
except PackageNotFoundError:
    torchao_version = "not installed"
print("torchao (optional, bypassed for PEFT LoRA):", torchao_version)


In [ ]:
# Đọc Kaggle Secrets trên Kaggle hoặc Colab Secrets trên Colab; không in hay ghi credential vào notebook.
import os
from pathlib import Path

if Path("/kaggle/working").exists():
    from kaggle_secrets import UserSecretsClient
    get_secret = UserSecretsClient().get_secret
else:
    from google.colab import userdata
    get_secret = userdata.get

os.environ["KAGGLE_USERNAME"] = get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = get_secret("KAGGLE_KEY")
del get_secret


## 1. Dữ liệu: tải và chia split

In [ ]:
# Giữ split đã chuẩn bị khi chạy lại notebook trong cùng Kaggle session.
import json
import subprocess
import sys
from pathlib import Path

split_dir = Path("data/splits")
meta_path = split_dir / "meta.json"
ready = meta_path.is_file() and all((split_dir / f"{name}.jsonl").is_file() for name in ("train", "val", "test"))
if ready:
    counts = json.loads(meta_path.read_text(encoding="utf-8"))["counts"]
    ready = sum(1 for _ in Path("data/images").glob("*.jpg")) == sum(item["images"] for item in counts.values())
if ready:
    print("Using existing data/splits and images")
else:
    subprocess.run([sys.executable, "scripts/prepare_data.py", "--kaggle"], check=True)


In [ ]:
# Kiểm tra split, thống kê box và tạo tập val 40 ảnh để chọn setup E4.
import json
import random
import statistics
from pathlib import Path

from scripts.labcommon import CLASSES, read_jsonl, write_jsonl

dataset_meta = json.loads(Path("data/splits/meta.json").read_text(encoding="utf-8"))
split_rows = {
    split: read_jsonl(Path("data/splits") / f"{split}.jsonl")
    for split in ("train", "val", "test")
}
session_sets = {split: {row["session"] for row in rows} for split, rows in split_rows.items()}
assert all(
    not (session_sets[a] & session_sets[b])
    for a, b in (("train", "val"), ("train", "test"), ("val", "test"))
)
print("dataset_version:", dataset_meta["dataset_version"])
box_areas = {label: [] for label in CLASSES}
for split, counts in dataset_meta["counts"].items():
    print(f"{split}: {counts['images']} ảnh; boxes={counts['boxes']}")
for row in split_rows["train"]:
    for obj in row["objects"]:
        box_areas[obj["label"]].append(obj["yolo"][2] * obj["yolo"][3])
median_box_area_pct = {
    label: 100 * statistics.median(areas) if areas else 0.0
    for label, areas in box_areas.items()
}
print("Train median box area (% ảnh):", median_box_area_pct)
stats = {
    "dataset_version": dataset_meta["dataset_version"],
    "counts": dataset_meta["counts"],
    "train_median_box_area_pct": median_box_area_pct,
}
stats_path = Path("report/evidence/dataset_stats.json")
stats_path.parent.mkdir(parents=True, exist_ok=True)
stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")

val_rows = split_rows["val"]
val_subset = random.Random(42).sample(val_rows, min(40, len(val_rows)))
val_debug_splits = Path("work/val_subset/splits")
val_debug_splits.mkdir(parents=True, exist_ok=True)
write_jsonl(val_debug_splits / "val.jsonl", val_subset)
(val_debug_splits / "meta.json").write_text(
    json.dumps(dataset_meta, ensure_ascii=False, indent=2), encoding="utf-8"
)
VAL_DEBUG_SPLITS = str(val_debug_splits)
print(f"Validation subset for E4 selection: {len(val_subset)} images")


In [ ]:
# Check the separate GPU worker before the long E0 inference and training runs.
worker_script = PROJECT_ROOT / "parallel_worker.py"
if not worker_script.is_file():
    raise FileNotFoundError(f"Update the repository clone: {worker_script} is missing")
visible_devices = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1").split(",")
if len(visible_devices) != 2 or torch.cuda.device_count() != 2:
    raise RuntimeError("Parallel run requires both Kaggle T4 GPUs")
worker_environment = os.environ.copy()
worker_environment.pop("KAGGLE_USERNAME", None)
worker_environment.pop("KAGGLE_KEY", None)
worker_environment["CUDA_VISIBLE_DEVICES"] = visible_devices[0]
subprocess.run(
    [sys.executable, "-u", str(worker_script), "--check"],
    cwd=PROJECT_ROOT, env=worker_environment, check=True,
)


## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# Chuyển nhãn YOLO chuẩn hoá sang bbox_2d nguyên trên thang 0–1000.
import json

def yolo_to_box(label, yolo):
    xc, yc, width, height = yolo
    coords = [
        round((xc - width / 2) * 1000),
        round((yc - height / 2) * 1000),
        round((xc + width / 2) * 1000),
        round((yc + height / 2) * 1000),
    ]
    return {"label": label, "bbox_2d": [max(0, min(1000, int(v))) for v in coords]}

def objects_to_answer(objects):
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda item: (item["bbox_2d"][1], item["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

assert yolo_to_box("pothole", [0.5, 0.5, 0.2, 0.4])["bbox_2d"] == [400, 300, 600, 700]
assert objects_to_answer([]) == "[]"


## 3. E0: zero-shot

In [ ]:
# E0: Qwen3-VL zero-shot trên test; ghi raw output để script của đề tự parse/chấm.
import gc
import json
import time
from pathlib import Path

import torch
import yaml
from bitsandbytes.nn import Linear4bit
from PIL import Image
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

config = yaml.safe_load(Path("configs/e0.yaml").read_text(encoding="utf-8"))
model_id = config["model_id"]
prompt = (
    "Detect every visible road-damage object. Classes: pothole, crack, manhole. "
    "Return only a JSON list; each item must have keys \"label\" and \"bbox_2d\". "
    "bbox_2d is [x1, y1, x2, y2] as integers on a 0-1000 scale relative to image width and height. "
    "Use [] if there are no objects. Do not add any other text."
)
batch_size = 4
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["model.visual", "lm_head"],
)
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id, dtype=torch.float16, device_map={"": 0}, quantization_config=quantization_config
)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "left"
model.eval()

named_params = list(model.named_parameters())
quantized_modules = [(name, module) for name, module in model.named_modules() if isinstance(module, Linear4bit)]
unexpected_quantized = [name for name, _ in quantized_modules if "language_model." not in name]
if unexpected_quantized:
    raise RuntimeError(f"Only LLM layers may be 4-bit; found {unexpected_quantized[:5]}")
packed_correction = sum(module.in_features * module.out_features - module.weight.numel() for _, module in quantized_modules)
visual_params = sum(p.numel() for name, p in named_params if "visual." in name)
projector_params = sum(
    p.numel() for name, p in named_params
    if "visual.merger" in name or "visual.deepstack_merger_list" in name
)
if projector_params < 100_000_000:
    raise RuntimeError(f"Vision projector is still quantized ({projector_params:,} parameters); expected about 109M")
llm_params = sum(
    p.numel() for name, p in named_params
    if "language_model." in name or "lm_head" in name
)
parameter_counts = {
    "vit": visual_params - projector_params,
    "projector": projector_params,
    "llm_embedding_head": llm_params + packed_correction,
    "total": sum(p.numel() for _, p in named_params) + packed_correction,
}
print(
    "Parameter counts:",
    f"ViT={parameter_counts['vit']:,}",
    f"projector={parameter_counts['projector']:,}",
    f"LLM+embedding/head={parameter_counts['llm_embedding_head']:,}",
    f"total={parameter_counts['total']:,}",
)
params_path = Path("report/evidence/model_parameters.json")
params_path.parent.mkdir(parents=True, exist_ok=True)
params_path.write_text(json.dumps(parameter_counts, indent=2), encoding="utf-8")
del named_params, quantized_modules

torch.cuda.reset_peak_memory_stats(0)
entries = [
    json.loads(line)
    for line in Path("data/splits/test.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
pred_path = Path("work/e0.jsonl")
pred_path.parent.mkdir(parents=True, exist_ok=True)
sample_outputs = []
torch.cuda.synchronize()
start_time = time.perf_counter()
with pred_path.open("w", encoding="utf-8") as out_file:
    for start in range(0, len(entries), batch_size):
        batch = entries[start : start + batch_size]
        images = []
        for entry in batch:
            with Image.open(PROJECT_ROOT / "data" / entry["image_path"]) as image:
                images.append(image.convert("RGB"))
        conversations = [
            [{"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ]}]
            for image in images
        ]
        inputs = processor.apply_chat_template(
            conversations, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors="pt", processor_kwargs={"padding": True}
        ).to(torch.device("cuda:0"))
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=config["max_new_tokens"], do_sample=False)
        prompt_length = inputs.input_ids.shape[1]
        raw_outputs = processor.batch_decode(
            generated[:, prompt_length:].cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        for entry, raw in zip(batch, raw_outputs):
            row = {"image_id": entry["image_id"], "raw": raw}
            out_file.write(json.dumps(row, ensure_ascii=False) + "\n")
            if len(sample_outputs) < 5:
                sample_outputs.append(row)
        out_file.flush()
        if (start // batch_size + 1) % 10 == 0 or start + len(batch) == len(entries):
            print(f"Generated {start + len(batch)}/{len(entries)} test images")
        del images, conversations, inputs, generated, raw_outputs

torch.cuda.synchronize()
inference_minutes = (time.perf_counter() - start_time) / 60
peak_vram_gb = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
print(f"Saved {len(entries)} predictions to {pred_path}")
print(f"Inference time: {inference_minutes:.2f} min; peak allocated VRAM: {peak_vram_gb:.2f} GB")
print("First five raw outputs:")
for row in sample_outputs:
    print(row["image_id"], row["raw"])
del model, processor
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# Chấm E0 đúng thang và tính thêm metric đối chứng khi nhầm output 0–1000 là pixel.
import subprocess
import sys

subprocess.run([
    sys.executable, "scripts/score.py",
    "--pred", "work/e0.jsonl", "--name", "E0",
    "--out", "report/evidence/e0.json",
    "--meta", "vision_precision=fp16",
    "--meta", f"peak_vram_gb={peak_vram_gb:.3f}",
    "--meta", f"inference_minutes={inference_minutes:.3f}",
], check=True)
subprocess.run([
    sys.executable, "scripts/score.py",
    "--pred", "work/e0.jsonl", "--name", "E0-wrong-pixel",
    "--coord", "pixel", "--out", "report/evidence/e0_wrong_coord_pixel.json",
], check=True)


## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# Encode ví dụ multimodal và chỉ tính loss trên token câu trả lời.
from pathlib import Path

import torch
from PIL import Image
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from transformers import AutoProcessor

from scripts.labcommon import read_jsonl

processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

def encode_example(image, prompt_text, answer):
    user_message = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt_text},
    ]}]
    prompt_encoding = processor.apply_chat_template(
        user_message, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors="pt"
    )
    full_encoding = processor.apply_chat_template(
        user_message + [{"role": "assistant", "content": answer}],
        tokenize=True, add_generation_prompt=False,
        return_dict=True, return_tensors="pt"
    )
    prefix_len = prompt_encoding["input_ids"].shape[1]
    if not torch.equal(
        full_encoding["input_ids"][:, :prefix_len], prompt_encoding["input_ids"]
    ):
        raise ValueError("Prompt encoding is not a prefix of the full conversation")
    labels = full_encoding["input_ids"].clone()
    labels[:, :prefix_len] = -100
    labels[full_encoding["attention_mask"] == 0] = -100
    encoded = {key: value for key, value in full_encoding.items() if torch.is_tensor(value)}
    encoded["labels"] = labels
    return encoded

class RoadDamageDataset(Dataset):
    def __init__(self, rows, image_root, prompt_text):
        self.rows = rows
        self.image_root = Path(image_root)
        self.prompt_text = prompt_text

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(self.image_root / row["image_path"]) as image:
            image = image.convert("RGB")
        return image, self.prompt_text, objects_to_answer(row["objects"])

def collate_examples(samples):
    encoded = []
    for image, prompt_text, answer in samples:
        try:
            item = encode_example(image, prompt_text, answer)
        finally:
            image.close()
        for key in ("input_ids", "attention_mask", "mm_token_type_ids", "labels"):
            if key in item:
                item[key] = item[key][0]
        encoded.append(item)
    pad_id = processor.tokenizer.pad_token_id
    batch = {
        "input_ids": pad_sequence(
            [item["input_ids"] for item in encoded], batch_first=True, padding_value=pad_id
        ),
        "attention_mask": pad_sequence(
            [item["attention_mask"] for item in encoded], batch_first=True, padding_value=0
        ),
        "labels": pad_sequence(
            [item["labels"] for item in encoded], batch_first=True, padding_value=-100
        ),
    }
    if "mm_token_type_ids" in encoded[0]:
        batch["mm_token_type_ids"] = pad_sequence(
            [item["mm_token_type_ids"] for item in encoded], batch_first=True, padding_value=0
        )
    for key in ("pixel_values", "image_grid_thw"):
        batch[key] = torch.cat([item[key] for item in encoded], dim=0)
    return batch

train_rows_for_check = read_jsonl("data/splits/train.jsonl")
sample_row = train_rows_for_check[0]
with Image.open(PROJECT_ROOT / "data" / sample_row["image_path"]) as sample_image:
    sample_encoding = encode_example(
        sample_image.convert("RGB"), prompt, objects_to_answer(sample_row["objects"])
    )
vision_token_count = (
    int((sample_encoding["mm_token_type_ids"] == 1).sum())
    if "mm_token_type_ids" in sample_encoding else -1
)
answer_tokens = sample_encoding["labels"][0]
answer_tokens = answer_tokens[answer_tokens != -100]
print("Image token count (640x360; GUIDE expects 220):", vision_token_count)
assert vision_token_count == 220, f"Expected 220 image tokens, found {vision_token_count}"
print("Masked target:", processor.tokenizer.decode(answer_tokens, skip_special_tokens=False))
del sample_encoding, answer_tokens, train_rows_for_check


## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# QLoRA setup, vòng train, suy luận và scoring dùng chung cho E1–E4.
import gc
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from pathlib import Path
from unittest.mock import patch
from PIL import Image
from scripts.labcommon import read_jsonl

import torch
import yaml
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration, get_cosine_schedule_with_warmup

LLM_TARGET_PATTERN = (
    r".*language_model\.layers\.\d+\."
    r"(?:self_attn\.(?:q_proj|k_proj|v_proj|o_proj)|mlp\.(?:gate_proj|up_proj|down_proj))"
)
VIT_TARGET_PATTERN = (
    r".*visual\.blocks\.\d+\."
    r"(?:attn\.(?:qkv|proj)|mlp\.(?:linear_fc1|linear_fc2))"
)

def load_training_model(model_id):
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        llm_int8_skip_modules=["model.visual", "lm_head"],
    )
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        model_id, dtype=torch.float16, device_map={"": 0}, quantization_config=quantization
    )
    projector_params = sum(
        p.numel() for name, p in model.named_parameters()
        if "visual.merger" in name or "visual.deepstack_merger_list" in name
    )
    if projector_params < 100_000_000:
        raise RuntimeError(f"Vision projector is still quantized ({projector_params:,} parameters); expected about 109M")
    return model

def apply_setup(model, setup):
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.enable_input_require_grads()
    target_pattern = LLM_TARGET_PATTERN
    if setup["lora"]["targets"] == "llm+vit":
        target_pattern = f"(?:{LLM_TARGET_PATTERN}|{VIT_TARGET_PATTERN})"
    matched_modules = [
        name for name, _ in model.named_modules() if re.fullmatch(target_pattern, name)
    ]
    expected = 348 if setup["lora"]["targets"] == "llm+vit" else 252
    if len(matched_modules) != expected:
        raise RuntimeError(f"{setup['name']}: expected {expected} LoRA targets, found {len(matched_modules)}")
    # ViT is fp16 and LLM uses bitsandbytes; Kaggle's old optional torchao must not gate LoRA.
    with patch("peft.tuners.lora.torchao.is_torchao_available", return_value=False):
        model = get_peft_model(model, LoraConfig(
            r=setup["lora"]["r"],
            lora_alpha=setup["lora"]["alpha"],
            lora_dropout=setup["lora"]["dropout"],
            target_modules=target_pattern,
            task_type="CAUSAL_LM",
        ))
    projector_parameters = 0
    if setup["train_projector"]:
        for name, parameter in model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                if not parameter.is_floating_point():
                    raise RuntimeError(f"{setup['name']}: quantized projector parameter {name} cannot be trained")
                parameter.data = parameter.data.float()
                parameter.requires_grad_(True)
                projector_parameters += parameter.numel()
        if projector_parameters == 0:
            raise RuntimeError(f"{setup['name']}: projector parameters were not found")
    for parameter in model.parameters():
        if parameter.requires_grad:
            parameter.data = parameter.data.float()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(
        f"{setup['name']}: LoRA targets={len(matched_modules)}, "
        f"projector trainable={projector_parameters:,}, trainable_params={trainable_params:,}"
    )
    return model, len(matched_modules), trainable_params


## 6. Vòng huấn luyện

In [ ]:
def save_run_checkpoint(run_dir, model, run_meta, step, loss_value):
    run_dir = Path(run_dir)
    adapter_dir = run_dir / "adapter"
    model.save_pretrained(adapter_dir, safe_serialization=True)
    projector_state = {
        name: parameter.detach().cpu()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
        and ("visual.merger" in name or "visual.deepstack_merger_list" in name)
    }
    if projector_state:
        temporary_path = run_dir / "projector.tmp.pt"
        torch.save(projector_state, temporary_path)
        os.replace(temporary_path, run_dir / "projector.pt")
    saved = {
        **run_meta,
        "global_step": step,
        "last_loss": loss_value,
    }
    (run_dir / "run.json").write_text(
        json.dumps(saved, ensure_ascii=False, indent=2), encoding="utf-8"
    )

def predict_split(model, rows, pred_path):
    processor.tokenizer.padding_side = "left"
    pred_path = Path(pred_path)
    pred_path.parent.mkdir(parents=True, exist_ok=True)
    with pred_path.open("w", encoding="utf-8") as out_file:
        for start in range(0, len(rows), 4):
            batch = rows[start : start + 4]
            images = []
            for entry in batch:
                with Image.open(PROJECT_ROOT / "data" / entry["image_path"]) as image:
                    images.append(image.convert("RGB"))
            conversations = [
                [{"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ]}]
                for image in images
            ]
            inputs = processor.apply_chat_template(
                conversations, tokenize=True, add_generation_prompt=True,
                return_dict=True, return_tensors="pt", processor_kwargs={"padding": True}
            ).to(torch.device("cuda:0"))
            with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
                generated = model.generate(
                    **inputs, max_new_tokens=config["max_new_tokens"], do_sample=False
                )
            prompt_length = inputs.input_ids.shape[1]
            raw_outputs = processor.batch_decode(
                generated[:, prompt_length:].cpu(),
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )
            for entry, raw in zip(batch, raw_outputs):
                out_file.write(json.dumps(
                    {"image_id": entry["image_id"], "raw": raw}, ensure_ascii=False
                ) + "\n")
            out_file.flush()
            if (start // 4 + 1) % 10 == 0 or start + len(batch) == len(rows):
                print(f"Predicted {start + len(batch)}/{len(rows)} images -> {pred_path}")
            del images, conversations, inputs, generated, raw_outputs

def score_prediction(pred_path, name, evidence_path, split="test", splits_dir="data/splits", meta_values=()):
    command = [
        sys.executable, "scripts/score.py",
        "--pred", str(pred_path), "--name", name, "--out", str(evidence_path),
        "--split", split, "--splits-dir", str(splits_dir),
    ]
    for value in meta_values:
        command.extend(["--meta", value])
    subprocess.run(command, check=True)
    return json.loads(Path(evidence_path).read_text(encoding="utf-8"))

def run_experiment(experiment, train_fraction=1.0, run_tag=None, evaluate_val=True, meta_extra=()):
    run_tag = run_tag or experiment
    setup = yaml.safe_load(Path(f"configs/{experiment}.yaml").read_text(encoding="utf-8"))
    seed = setup["seed"]
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    processor.tokenizer.padding_side = "right"
    train_rows = read_jsonl("data/splits/train.jsonl")
    random.Random(seed).shuffle(train_rows)
    if train_fraction < 1.0:
        train_rows = train_rows[:max(1, int(len(train_rows) * train_fraction))]
    dataset = RoadDamageDataset(train_rows, PROJECT_ROOT / "data", prompt)
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        dataset,
        batch_size=setup["batch_size"],
        shuffle=True,
        generator=generator,
        collate_fn=collate_examples,
        num_workers=0,
        pin_memory=True,
        drop_last=True,
    )
    run_dir = Path("work") / f"{run_tag}_fp16vision"
    run_dir.mkdir(parents=True, exist_ok=True)
    model = None
    try:
        model = load_training_model(setup["model_id"])
        model, matched_count, trainable_params = apply_setup(model, setup)
        resume_meta = None
        checkpoint_metadata_path = run_dir / "run.json"
        if checkpoint_metadata_path.exists():
            candidate = json.loads(checkpoint_metadata_path.read_text(encoding="utf-8"))
            if candidate.get("vision_precision") != "fp16" or candidate.get("trainable_params") != trainable_params:
                raise RuntimeError(f"{setup['name']}: checkpoint model layout differs; refusing to reuse it")
            if candidate.get("global_step") == setup["max_steps"]:
                adapter_path = run_dir / "adapter" / "adapter_model.safetensors"
                projector_path = run_dir / "projector.pt"
                if not adapter_path.is_file() or (setup["train_projector"] and not projector_path.is_file()):
                    raise FileNotFoundError(f"{setup['name']}: completed checkpoint is missing model weights")
                from safetensors.torch import load_file
                from peft.utils.save_and_load import set_peft_model_state_dict
                adapter_result = set_peft_model_state_dict(model, load_file(str(adapter_path)))
                if adapter_result.unexpected_keys:
                    raise RuntimeError(f"{setup['name']}: unexpected adapter checkpoint keys: {adapter_result.unexpected_keys[:5]}")
                if setup["train_projector"]:
                    projector_state = torch.load(projector_path, map_location="cpu", weights_only=True)
                    unknown_keys = set(projector_state).difference(model.state_dict())
                    if unknown_keys:
                        raise RuntimeError(f"{setup['name']}: unknown projector checkpoint keys: {sorted(unknown_keys)[:5]}")
                    model.load_state_dict(projector_state, strict=False)
                resume_meta = candidate
                print(f"{setup['name']}: restored completed checkpoint at step {candidate['global_step']}; skipping training")
            else:
                raise RuntimeError(f"{setup['name']}: incomplete checkpoint at step {candidate.get('global_step')}; refusing to retrain automatically")
        trainable = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(
            trainable, lr=setup["lr"], weight_decay=0.0
        )
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=max(1, int(setup["max_steps"] * 0.05)),
            num_training_steps=setup["max_steps"],
        )
        scaler = torch.amp.GradScaler("cuda")
        model.train()
        model.config.use_cache = False
        torch.cuda.reset_peak_memory_stats(0)
        data_iter = iter(loader)
        train_start = time.perf_counter()
        step_losses = []
        run_meta = {
            "setup": setup["name"],
            "vision_precision": "fp16",
            "base_setup": experiment,
            "train_fraction": train_fraction,
            "train_examples": len(train_rows),
            "trainable_params": trainable_params,
            "matched_lora_modules": matched_count,
        }
        if resume_meta is None:
            step = attempts = skipped_updates = consecutive_skips = 0
        else:
            step = setup["max_steps"]
            attempts = resume_meta.get("train_attempts", step)
            skipped_updates = resume_meta.get("overflow_skips", 0)
            consecutive_skips = 0
        while step < setup["max_steps"]:
            attempts += 1
            next_step = step + 1
            optimizer.zero_grad(set_to_none=True)
            micro_losses = []
            for _ in range(setup["grad_accum"]):
                try:
                    batch = next(data_iter)
                except StopIteration:
                    data_iter = iter(loader)
                    batch = next(data_iter)
                batch = {
                    key: value.to(torch.device("cuda:0"), non_blocking=True)
                    for key, value in batch.items()
                }
                with torch.autocast("cuda", dtype=torch.float16):
                    loss = model(**batch).loss
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"{setup['name']} loss is not finite at optimizer step {next_step}")
                micro_losses.append(float(loss.detach().float().item()))
                scaler.scale(loss / setup["grad_accum"]).backward()
            for parameter in trainable:
                if parameter.grad is not None and parameter.grad.dtype == torch.float16:
                    parameter.data = parameter.data.float()
                    parameter.grad = parameter.grad.float()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            if not torch.isfinite(grad_norm):
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped_updates += 1
                consecutive_skips += 1
                print(
                    f"{setup['name']}: skipped nonfinite gradient update "
                    f"({skipped_updates} total; scale {old_scale:g}->{scaler.get_scale():g})"
                )
                if consecutive_skips >= 10:
                    raise FloatingPointError(f"{setup['name']} had 10 consecutive nonfinite gradient updates")
                continue
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            consecutive_skips = 0
            step = next_step
            step_loss = sum(micro_losses) / len(micro_losses)
            step_losses.append(step_loss)
            if step % 25 == 0:
                print(f"{setup['name']} step {step}/{setup['max_steps']} loss={sum(step_losses[-25:]) / len(step_losses[-25:]):.4f}")
            if step % 100 == 0 or step == setup["max_steps"]:
                torch.cuda.synchronize()
                elapsed = (time.perf_counter() - train_start) / 60
                peak = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
                save_run_checkpoint(
                    run_dir, model,
                    {**run_meta, "train_minutes": elapsed, "peak_vram_gb": peak,
                     "overflow_skips": skipped_updates, "train_attempts": attempts},
                    step, step_loss,
                )
        torch.cuda.synchronize()
        if resume_meta is None:
            train_minutes = (time.perf_counter() - train_start) / 60
            peak_vram_gb = torch.cuda.max_memory_allocated(0) / (1024 ** 3)
        else:
            train_minutes = float(resume_meta["train_minutes"])
            peak_vram_gb = float(resume_meta["peak_vram_gb"])
        run_meta.update({"overflow_skips": skipped_updates, "train_attempts": attempts})
        model.gradient_checkpointing_disable()
        model.config.use_cache = True
        model.eval()
        meta_values = [
            f"trainable_params={trainable_params}",
            f"peak_vram_gb={peak_vram_gb:.6f}",
            f"train_minutes={train_minutes:.6f}",
            f"overflow_skips={skipped_updates}",
            f"data_size={train_fraction}",
            f"base_setup={experiment}",
            *meta_extra,
        ]
        run_info = {
            **run_meta,
            "train_minutes": train_minutes,
            "peak_vram_gb": peak_vram_gb,
            "run_dir": str(run_dir),
            "pred_path": str(Path("work") / f"{run_tag}.jsonl"),
        }
        if evaluate_val:
            val_rows = read_jsonl(Path(VAL_DEBUG_SPLITS) / "val.jsonl")
            val_pred = Path("work/val_subset") / f"{run_tag}.jsonl"
            predict_split(model, val_rows, val_pred)
            run_info["val_score"] = score_prediction(
                val_pred, f"{setup['name']}-val40",
                Path("work/val_subset") / f"{run_tag}.json",
                split="val", splits_dir=VAL_DEBUG_SPLITS, meta_values=meta_values,
            )
        test_rows = read_jsonl("data/splits/test.jsonl")
        test_pred = Path("work") / f"{run_tag}.jsonl"
        predict_split(model, test_rows, test_pred)
        run_info["test_score"] = score_prediction(
            test_pred, setup["name"] if run_tag == experiment else run_tag,
            Path("report/evidence") / f"{run_tag}.json",
            split="test", meta_values=meta_values,
        )
        if run_info["test_score"]["missing_predictions"] != 0:
            raise RuntimeError(f"{run_tag}: scorer found missing test predictions")
        (run_dir / "run.json").write_text(
            json.dumps({
                **run_meta,
                "global_step": setup["max_steps"],
                "train_minutes": train_minutes,
                "peak_vram_gb": peak_vram_gb,
                "val_f1": run_info.get("val_score", {}).get("micro", {}).get("f1"),
                "test_f1": run_info["test_score"]["micro"]["f1"],
            }, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        return run_info
    finally:
        if model is not None:
            del model
        gc.collect()
        torch.cuda.empty_cache()


## 7. Suy luận và chấm E1–E3

In [ ]:
# Fail fast on E3 setup before E1/E2 training.
e3_setup = yaml.safe_load(Path("configs/e3.yaml").read_text(encoding="utf-8"))
e3_probe = None
try:
    e3_probe = load_training_model(e3_setup["model_id"])
    e3_probe, _, _ = apply_setup(e3_probe, e3_setup)
finally:
    del e3_probe
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# Run E1/E2 concurrently, then start E3 on whichever GPU becomes free first.
experiment_runs = {}
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Expected 2 Kaggle GPUs, found {torch.cuda.device_count()}")
worker_script = PROJECT_ROOT / "parallel_worker.py"
if not worker_script.is_file():
    raise FileNotFoundError(f"Parallel worker is missing: {worker_script}")
visible_devices = os.environ.get("CUDA_VISIBLE_DEVICES", "0,1").split(",")
if len(visible_devices) != 2:
    raise RuntimeError(f"Expected 2 visible GPU IDs, found {visible_devices}")

def launch_experiment(name, gpu_index):
    log_path = Path("work") / f"{name}_gpu{gpu_index}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_handle = log_path.open("w", encoding="utf-8")
    environment = os.environ.copy()
    environment.pop("KAGGLE_USERNAME", None)
    environment.pop("KAGGLE_KEY", None)
    environment["CUDA_VISIBLE_DEVICES"] = visible_devices[gpu_index]
    process = subprocess.Popen(
        [sys.executable, "-u", str(worker_script), name],
        cwd=PROJECT_ROOT, env=environment, stdout=log_handle, stderr=subprocess.STDOUT,
    )
    print(f"Started {name.upper()} on GPU {gpu_index}; log: {log_path}")
    return {"name": name, "process": process, "handle": log_handle, "log": log_path, "offset": 0}

def show_new_log(job):
    with job["log"].open("r", encoding="utf-8", errors="replace") as reader:
        reader.seek(job["offset"])
        chunk = reader.read()
        job["offset"] = reader.tell()
    if chunk:
        print(f"[{job['name'].upper()}] {chunk}", end="", flush=True)
    return chunk

pending_experiments = ["e1", "e2", "e3"]
active_jobs = {}
try:
    active_jobs[0] = launch_experiment(pending_experiments.pop(0), 0)
    # Let the first model finish loading before starting the second copy.
    first_setup_log = ""
    while True:
        first_setup_log += show_new_log(active_jobs[0])
        if "E1: LoRA targets=" in first_setup_log:
            break
        if active_jobs[0]["process"].poll() is not None:
            raise RuntimeError(f"E1 failed during setup; see {active_jobs[0]['log']}")
        time.sleep(2)
    active_jobs[1] = launch_experiment(pending_experiments.pop(0), 1)
    while active_jobs:
        for gpu_index, job in list(active_jobs.items()):
            show_new_log(job)
            exit_code = job["process"].poll()
            if exit_code is None:
                continue
            job["handle"].close()
            show_new_log(job)
            del active_jobs[gpu_index]
            if exit_code != 0:
                raise RuntimeError(f"{job['name'].upper()} failed (exit {exit_code}); see {job['log']}")
            result_path = Path("work") / f"{job['name']}_result.json"
            if not result_path.is_file():
                raise FileNotFoundError(f"{job['name'].upper()} finished without {result_path}")
            experiment_runs[job["name"]] = json.loads(result_path.read_text(encoding="utf-8"))
            print(f"Completed {job['name'].upper()} on GPU {gpu_index}")
            if pending_experiments:
                active_jobs[gpu_index] = launch_experiment(pending_experiments.pop(0), gpu_index)
        if active_jobs:
            time.sleep(5)
finally:
    for job in active_jobs.values():
        if job["process"].poll() is None:
            job["process"].terminate()
            job["process"].wait()
        job["handle"].close()


### E2 · E1 + trainable projector


In [ ]:
print("E2 test F1:", experiment_runs["e2"]["test_score"]["micro"]["f1"])


### E3 · E2 + ViT attention/MLP LoRA


In [ ]:
print("E3 test F1:", experiment_runs["e3"]["test_score"]["micro"]["f1"])


## 8. E4 · Data-size ablation (10% and 100%)


In [ ]:
# E4: Compare 10% vs 100% training data using the setup with the best validation-40 F1.
import json
import random
import zipfile
from pathlib import Path
from PIL import Image, ImageDraw
from scripts.labcommon import CLASSES, read_jsonl
from scripts.score import parse_output

if any(experiment_runs[key].get("vision_precision") != "fp16" for key in ("e1", "e2", "e3")):
    raise RuntimeError("E4 requires E1–E3 results from the corrected full-precision vision setup")
legacy_selection = Path("report/evidence/e4_selection.json")
if legacy_selection.exists():
    legacy_selection.replace("report/evidence/selection_e4.json")

val_f1 = {
    key: experiment_runs[key]["val_score"]["micro"]["f1"]
    for key in ("e1", "e2", "e3")
}
best_setup = max(val_f1, key=val_f1.get)
best_score = experiment_runs[best_setup]["test_score"]
full_path = Path("report/evidence") / f"{best_setup}.json"
full_result = json.loads(full_path.read_text(encoding="utf-8"))
full_result["name"] = f"E4 data_size=1.0 (reused {best_setup})"
full_result["meta"].update({
    "data_size": 1.0,
    "base_setup": best_setup,
    "e4_variable": "data_size",
    "e4_value": 1.0,
})
Path("report/evidence/e4_data_size_1.0.json").write_text(
    json.dumps(full_result, ensure_ascii=False, indent=2), encoding="utf-8"
)
Path("report/evidence/selection_e4.json").write_text(
    json.dumps({"selected_by": "validation subset of 40 images", "best_setup": best_setup,
                "val_f1": val_f1, "reused_test_f1": best_score["micro"]["f1"]},
               ensure_ascii=False, indent=2), encoding="utf-8"
)

worst_class = min(CLASSES, key=lambda label: best_score["per_class"][label]["f1"])
dataset_stats = json.loads(Path("report/evidence/dataset_stats.json").read_text(encoding="utf-8"))
test_rows = read_jsonl("data/splits/test.jsonl")
pred_rows = read_jsonl(Path("work") / f"{best_setup}.jsonl")
pred_by_id = {row["image_id"]: row["raw"] for row in pred_rows}
candidates = [row for row in test_rows if any(obj["label"] == worst_class for obj in row["objects"])]
if not candidates:
    raise RuntimeError(f"No test images found for worst class {worst_class}")
examples = random.Random(42).sample(candidates, min(6, len(candidates)))
figure_dir = Path("report/figures")
figure_dir.mkdir(parents=True, exist_ok=True)
thumb_w, thumb_h = 640, 360
caption_h, cols = 28, 2
rows_n = (len(examples) + cols - 1) // cols
sheet = Image.new("RGB", (cols * thumb_w, rows_n * (thumb_h + caption_h)), "white")
for index, entry in enumerate(examples):
    with Image.open(Path(PROJECT_ROOT) / "data" / entry["image_path"]) as source:
        image = source.convert("RGB").resize((thumb_w, thumb_h))
    draw = ImageDraw.Draw(image)
    for obj in entry["objects"]:
        xc, yc, width, height = obj["yolo"]
        box = ((xc - width / 2) * thumb_w, (yc - height / 2) * thumb_h,
               (xc + width / 2) * thumb_w, (yc + height / 2) * thumb_h)
        draw.rectangle(box, outline="#00A650", width=3)
        draw.text((box[0], max(0, box[1] - 14)), f"GT {obj['label']}", fill="#00A650")
    parsed = parse_output(pred_by_id.get(entry["image_id"], ""))
    for obj in parsed or []:
        x1, y1, x2, y2 = obj["bbox"]
        box = (x1 * thumb_w / 1000, y1 * thumb_h / 1000,
               x2 * thumb_w / 1000, y2 * thumb_h / 1000)
        draw.rectangle(box, outline="#E63946", width=3)
        draw.text((box[0], min(thumb_h - 14, box[3])), f"P {obj['label']}", fill="#E63946")
    x, y = (index % cols) * thumb_w, (index // cols) * (thumb_h + caption_h)
    sheet.paste(image, (x, y + caption_h))
    ImageDraw.Draw(sheet).text((x + 4, y + 5), f"{entry['image_id']} | GT green, prediction red", fill="black")
figure_path = figure_dir / f"q9_{best_setup}_examples.png"
sheet.save(figure_path)
train_counts = dataset_stats["counts"]["train"]["boxes"]
test_counts = dataset_stats["counts"]["test"]["boxes"]
Path("report/evidence/q9_analysis.json").write_text(
    json.dumps({"best_setup_by_val40": best_setup, "worst_class": worst_class,
                "class_test_score": best_score["per_class"][worst_class],
                "train_box_count": train_counts[worst_class],
                "test_box_count": test_counts[worst_class],
                "train_median_box_area_pct": dataset_stats["train_median_box_area_pct"][worst_class],
                "example_image_ids": [row["image_id"] for row in examples],
                "figure": str(figure_path)}, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("E4 base selected by val-40:", best_setup, val_f1)
print("Worst class for Q9:", worst_class, "figure:", figure_path)
experiment_runs["e4_data_size_0.1"] = run_experiment(
    best_setup, train_fraction=0.1, run_tag="e4_data_size_0.1",
    meta_extra=("e4_variable=data_size", "e4_value=0.1"),
)

# Bundle only report evidence and raw predictions; omit large training checkpoints.
archive_path = Path(PROJECT_ROOT).parent / "day10_report_outputs.zip"
with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for pattern in ("report/evidence/*.json", "report/figures/*.png", "work/*.jsonl",
                    "work/val_subset/*.json", "work/val_subset/*.jsonl",
                    "work/val_subset/splits/*.jsonl", "work/val_subset/splits/meta.json",
                    "work/*_fp16vision/run.json"):
        for path in sorted(Path(".").glob(pattern)):
            archive.write(path, path.as_posix())
print("Download from Kaggle output:", archive_path, f"({archive_path.stat().st_size / 1024:.1f} KiB)")
